# Twitter Sentiment Analysis of Apple and Google Products

Author: Dorcas Ndambuki.

## **1. BUSINESS UNDERSTANDING**
Apple and Google receive thousands of tweets discussing their products. Monitoring customer sentiment manually is inefficient and difficult to scale.

# 1.1.Business Context
 Apple and Google operate in a highly competitive, fast-paced consumer tech market. Customer sentiment on social media shifts rapidly in response to product launches, software updates, and public relations events.

# 1.2 Business Problem
Apple and Google receive thousands of Tweets/public opinions daily discussing their products and services on Social Media. Brand managers cannot manually monitor the thousands of tweets as is The goal of this project is to build a machine learning model capable of automatically classifying tweet sentiment as positive, negative, or neutral.


# 1.3 Stakeholders
These are the people who will use or benefit from the results of this model ie Business stakeholders use the results to make decisions,Technical Stakeholders who build, deploy, and maintain the data systems and infrastructure or End-Users who interact with the model's output on a daily basis to do their jobs.

Brand Managers (Apple & Google)- Protect and grow the reputation of specific product lines.

Marketing Executives-  Design advertising campaigns and launch new products.

Customer Experience Executives-Oversee the entire customer journey and ensure overall satisfaction.

Business Executives-Make high-level strategic decisions, allocate budgets, and report to investors.

Product Support Teams-Resolve technical issues and assist customers directly.



# 1.4 Business Questions

Key Questions

1.Can tweet sentiment be predicted accurately?

2.Which words most strongly drive sentiment?

3.What themes appear in negative feedback?

4.How can Apple and Google use these insights?

# 1.5 Success Metric Selection

* **Primary metric**
    * Macro Recall

* **Secondary metrics**
    * Macro F1
    * Accuracy

## **2. DATA UNDERSTANDING**
# 2.1 Data Source

The dataset used in this project is the **Twitter Product Sentiment Dataset**, originally compiled by CrowdFlower and made available via data.world. It contains Tweets related to Apple and Google products and each Tweet was labelled according to its sentiment.


# 2.2 Dataset Dimensions
Number of observations (rows) before cleaning: ~9,000+ Tweets and 3 columns sentimentally labelled.

# 2.3 Column names and Data Types

Tweet_text (Object/String): The raw text of the tweet ($9,092$ non-null values, $1$ missing value).

Emotion_in_tweet_is_directed_at (Object/String): The specific product or brand being targeted ($3,291$ non-null values, high missingness).

Is_there_an_emotion_directed_at_a_brand_or_product (Object/String): Our target label ($9,093$ non-null values).

# 2.4 Target Variable
* Positive

* Negative

* Neutral (or Neither)

## **3. DATA PREPARATION AND CLEANING**

# 3.1 Data Preparation

In [2]:
#Importing Libraries
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting aesthetics
sns.set_theme(style="whitegrid")
%matplotlib inline

# NLP preprocessing
import re
import string

# NLTK tools
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Reproducibility
import random


[nltk_data] Downloading package stopwords to C:\Users\Moringa
[nltk_data]     School\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Moringa
[nltk_data]     School\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Moringa
[nltk_data]     School\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

seed_everything()

# 2.4 Data Loading

In [8]:
# 1. Load dataset
df = pd.read_csv('data/twitter_sentiment.csv', encoding='unicode_escape')

In [9]:
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   tweet_text                                          9092 non-null   object
 1   emotion_in_tweet_is_directed_at                     3291 non-null   object
 2   is_there_an_emotion_directed_at_a_brand_or_product  9093 non-null   object
dtypes: object(3)
memory usage: 213.2+ KB


In [11]:
df.shape

(9093, 3)

In [12]:
df.value_counts

<bound method DataFrame.value_counts of                                              tweet_text  \
0     .@wesley83 I have a 3G iPhone. After 3 hrs twe...   
1     @jessedee Know about @fludapp ? Awesome iPad/i...   
2     @swonderlin Can not wait for #iPad 2 also. The...   
3     @sxsw I hope this year's festival isn't as cra...   
4     @sxtxstate great stuff on Fri #SXSW: Marissa M...   
...                                                 ...   
9088                      Ipad everywhere. #SXSW {link}   
9089  Wave, buzz... RT @mention We interrupt your re...   
9090  Google's Zeiger, a physician never reported po...   
9091  Some Verizon iPhone customers complained their...   
9092  Ï¡Ïàü_ÊÎÒ£Áââ_£â_ÛâRT @...   

     emotion_in_tweet_is_directed_at  \
0                             iPhone   
1                 iPad or iPhone App   
2                               iPad   
3                 iPad or iPhone App   
4                             Google   
...        